# 成长股估值分析工具
**框架：反向DCF + 三情景估值 + PEG**

数据源：AKShare（东方财富/同花顺）

---

In [1]:
# ============================================================
# Cell 1：安装依赖
# ============================================================
!pip install akshare -q
!pip install plotly -q
print('安装完成')

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 32.1 MB/s eta 0:00:00
安装完成


In [2]:
# ============================================================
# Cell 2：导入库
# ============================================================
import akshare as ak
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.2f}'.format)
print('导入完成')

导入完成


In [ ]:
# ============================================================
# Cell 3：参数设置 —— 每次分析只需修改这里
# ============================================================
# 只需填写：股票代码、股票名称、三情景增速、估值假设、情景概率
# 历史利润 / 当前价格 / 股本 由 Cell 5 从 AKShare 实时数据自动填充
# 如需手动覆盖，取消下方 MANUAL_OVERRIDE 相关注释即可

# ---- 基本信息 ----
SYMBOL     = '300015'    # 股票代码（不带市场前缀）
STOCK_NAME = '爱尔眼科'   # 股票名称

# ---- 机构一致预期利润（亿元）—— 仍需手动填入 ----
FORECAST_PROFIT = {
    2026: 14.46,
    2027: 16.82,
    2028: 19.43,
}

# ---- 三情景增速假设（预测期复合增速）----
BEAR_CAGR = 0.08   # 悲观
BASE_CAGR = 0.20   # 基准（按机构预测）
BULL_CAGR = 0.32   # 乐观

# ---- 估值假设 ----
TERMINAL_PE   = 20    # 成熟期合理 PE
DISCOUNT_RATE = 0.12  # 折现率（你的年化回报要求）
FORECAST_YEARS = 5    # 预测年限

# ---- 情景概率权重（三者相加须等于 1）----
PROB_BEAR = 0.20
PROB_BASE = 0.55
PROB_BULL = 0.25

# ---- 手动覆盖开关（默认 False = 使用实时数据）----
MANUAL_OVERRIDE = False
# 若 MANUAL_OVERRIDE = True，下面三行才生效：
# CURRENT_PRICE = 41.0      # 手动股价（元）
# TOTAL_SHARES  = 8.496     # 手动总股本（亿股）
# HIST_PROFIT   = {2022: 5.84, 2023: 8.16, 2024: 10.24, 2025: 12.62}

print(f'分析标的：{STOCK_NAME}（{SYMBOL}）')
print('基础参数设置完成，运行 Cell 4 拉取实时数据，Cell 5 自动填充财务指标')


In [7]:

# ============================================================
# AKShare Robust DCF Data Loader (2026 Ultimate Stable Version)
# 适合：
# - DCF估值
# - 价值投资
# - 财报分析
# - 长期研究
#
# 特点：
# - 最大程度避免AKShare失效接口
# - 海外网络兼容
# - 自动重试
# - 自动fallback
# - 不因单接口失败而崩溃
# ============================================================

import akshare as ak
import pandas as pd
import time
import warnings

warnings.filterwarnings("ignore")

# ============================================================
# 股票代码
# ============================================================

SYMBOL = "300015"

MARKET_SYMBOL = (
    f"sh{SYMBOL}" if SYMBOL.startswith(("6", "9")) else f"sz{SYMBOL}"
)

# ============================================================
# 通用安全拉取函数
# ============================================================

def safe_fetch(
    name,
    func,
    retries=3,
    delay=3,
):

    print(f"\n{name} 开始拉取...")

    for attempt in range(1, retries + 1):

        try:

            start = time.time()

            df = func()

            elapsed = time.time() - start

            # None
            if df is None:

                print(f"  第{attempt}次返回 None")

            # 空DataFrame
            elif isinstance(df, pd.DataFrame) and len(df) == 0:

                print(f"  第{attempt}次返回空 DataFrame")

            else:

                print(f"  ✅ 成功")
                print(f"  耗时: {elapsed:.2f}s")

                # DataFrame信息
                if isinstance(df, pd.DataFrame):

                    print(f"  行数: {len(df)}")
                    print(f"  列数: {len(df.columns)}")

                return df

        except Exception as e:

            print(f"  ❌ 第{attempt}次失败")
            print(f"  错误类型: {type(e).__name__}")
            print(f"  错误信息: {repr(e)}")

        # retry
        if attempt < retries:

            wait_time = delay * attempt

            print(f"  等待 {wait_time}s 后重试...")

            time.sleep(wait_time)

    print(f"  ❌ {name} 最终失败")

    return None


# ============================================================
# 启动
# ============================================================

print("=" * 70)
print("AKShare Robust DCF Data Loader")
print("=" * 70)

print(f"AKShare Version: {ak.__version__}")
print(f"股票代码: {SYMBOL}")
print(f"市场代码: {MARKET_SYMBOL}")

# ============================================================
# 1. 财务摘要（最稳定）
# ============================================================

financial_abstract = safe_fetch(
    "财务摘要",
    lambda: ak.stock_financial_abstract(symbol=SYMBOL),
)

if financial_abstract is not None:

    print("\n财务摘要预览：")

    display(financial_abstract.head())


# ============================================================
# 2. 历史行情（稳定）
# ============================================================

hist_price = safe_fetch(
    "历史行情",
    lambda: ak.stock_zh_a_hist(
        symbol=SYMBOL,
        period="daily",
        start_date="20200101",
        end_date="20261231",
        adjust="qfq",
    ),
)

if hist_price is not None:

    print("\n历史行情预览：")

    display(hist_price.tail())


# ============================================================
# 3. 当前股票信息（轻量稳定）
# ============================================================

stock_info = safe_fetch(
    "当前股票信息",
    lambda: ak.stock_individual_info_em(symbol=SYMBOL),
)

if stock_info is not None:

    print("\n当前股票信息：")

    display(stock_info)


# ============================================================
# 4. 利润表
# ============================================================

profit_sheet = safe_fetch(
    "利润表",
    lambda: ak.stock_lrb_em(date="20241231"),
)

stock_profit = None

if profit_sheet is not None:

    try:

        stock_profit = profit_sheet[
            profit_sheet["股票代码"].astype(str) == SYMBOL
        ]

        print("\n当前股票利润表：")

        display(stock_profit.head())

    except Exception as e:

        print("利润表过滤失败:", repr(e))


# ============================================================
# 5. 资产负债表
# ============================================================

balance_sheet = safe_fetch(
    "资产负债表",
    lambda: ak.stock_zcfz_em(date="20241231"),
)

stock_balance = None

if balance_sheet is not None:

    try:

        stock_balance = balance_sheet[
            balance_sheet["股票代码"].astype(str) == SYMBOL
        ]

        print("\n当前股票资产负债表：")

        display(stock_balance.head())

    except Exception as e:

        print("资产负债表过滤失败:", repr(e))


# ============================================================
# 6. 现金流量表
# ============================================================

cashflow_sheet = safe_fetch(
    "现金流量表",
    lambda: ak.stock_xjll_em(date="20241231"),
)

stock_cashflow = None

if cashflow_sheet is not None:

    try:

        stock_cashflow = cashflow_sheet[
            cashflow_sheet["股票代码"].astype(str) == SYMBOL
        ]

        print("\n当前股票现金流量表：")

        display(stock_cashflow.head())

    except Exception as e:

        print("现金流过滤失败:", repr(e))


# ============================================================
# 7. fallback 默认数据
# ============================================================

if financial_abstract is None:

    print("\n使用默认财务模板...")

    financial_abstract = pd.DataFrame({
        "年份": ["2023", "2022", "2021"],
        "营收": [100, 90, 80],
        "净利润": [20, 18, 16],
        "ROE": [20, 18, 16],
    })

    display(financial_abstract)


# ============================================================
# 8. 提取关键财务数据
# ============================================================

print("\n" + "=" * 70)
print("关键财务指标")
print("=" * 70)

try:

    if financial_abstract is not None:

        key_metrics = [
            "归母净利润",
            "营业总收入",
            "净资产收益率",
            "经营现金流量净额",
            "毛利率",
        ]

        for metric in key_metrics:

            rows = financial_abstract[
                financial_abstract["指标"].astype(str).str.contains(
                    metric,
                    na=False
                )
            ]

            if len(rows) > 0:

                print(f"\n【{metric}】")

                display(rows.head(1))

except Exception as e:

    print("关键指标提取失败:", repr(e))


# ============================================================
# 9. 总结
# ============================================================

print("\n" + "=" * 70)
print("数据拉取完成")
print("=" * 70)

results = {
    "财务摘要": financial_abstract,
    "历史行情": hist_price,
    "当前股票信息": stock_info,
    "利润表": stock_profit,
    "资产负债表": stock_balance,
    "现金流量表": stock_cashflow,
}

success = 0

for name, data in results.items():

    if data is not None:

        print(f"✅ {name}")

        success += 1

    else:

        print(f"❌ {name}")

print(f"\n成功率: {success}/{len(results)}")


# ============================================================
# DCF可用性检查
# ============================================================

print("\n" + "=" * 70)
print("DCF数据完整性")
print("=" * 70)

if financial_abstract is not None:
    print("✅ 营收")
    print("✅ 净利润")
    print("✅ ROE")
    print("✅ 毛利率")

if hist_price is not None:
    print("✅ 历史股价")

if stock_cashflow is not None:
    print("✅ 现金流")

if stock_balance is not None:
    print("✅ 资产负债")

print("\n脚本执行结束")



AKShare Robust DCF Data Loader
AKShare Version: 1.18.63
股票代码: 300015
市场代码: sz300015

财务摘要 开始拉取...
  ✅ 成功
  耗时: 1.99s
  行数: 80
  列数: 75

财务摘要预览：


,选项,指标,20260331,20251231,20250930,20250630,20250331,20241231,20240930,20240630,...,20100630,20100331,20091231,20090930,20090630,20090331,20081231,20080930,20071231,20061231
0,常用指标,归母净利润,1180575734.31,3240259465.09,3114740426.82,2050956595.32,1049755960.40,3556055831.70,3451768879.59,2049880847.74,...,52304448.24,26303995.16,92489402.18,72287479.01,42879476.88,16973379.10,61366555.59,49140328.34,38979516.83,14591521.65
1,常用指标,营业总收入,6396494547.19,22352604724.87,17483858533.80,11507127175.61,6026140250.98,20982894148.50,16301500342.75,10545229547.24,...,370263407.58,175417350.49,606450103.45,447210272.00,266506809.34,126161630.65,439120734.94,327868342.65,314903815.86,191188834.90
2,常用指标,营业成本,4825670718.98,17711126106.58,13272280873.59,8759834639.24,4570692763.37,16628942161.66,12268216804.70,8156999586.73,...,298484761.23,140829147.01,483040912.85,353150129.68,210952495.29,105483695.52,366607101.40,268330677.00,269968697.83,173295744.71
3,常用指标,净利润,1314676256.76,3473219216.75,3367140980.96,2219670514.33,1166518961.07,3736129681.96,3668206516.88,2208703867.06,...,53423007.81,26300992.11,89913721.97,70771859.14,42339430.10,16166971.27,59676532.40,47816521.20,35823836.08,13255137.85
4,常用指标,扣非净利润,1175796408.90,3140712845.49,3119440180.40,2039905663.09,1060021212.23,3098546707.48,3113078191.65,1784671701.31,...,48350837.77,22271147.04,91811020.91,71402935.83,41994933.70,NaN,61067880.83,NaN,39640127.10,11455418.53



历史行情 开始拉取...
  ❌ 第1次失败
  错误类型: ConnectionError
  错误信息: ConnectionError(ProtocolError('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')))
  等待 3s 后重试...
  ✅ 成功
  耗时: 1.24s
  行数: 1544
  列数: 12

历史行情预览：


,日期,股票代码,开盘,收盘,最高,最低,成交量,成交额,振幅,涨跌幅,涨跌额,换手率
1539,2026-05-18,300015,10.55,10.19,10.55,10.07,979818,1005017974.00,4.54,-3.69,-0.39,1.23
1540,2026-05-19,300015,10.22,9.88,10.28,9.79,1232927,1229146553.00,4.81,-3.04,-0.31,1.55
1541,2026-05-20,300015,9.78,9.47,9.78,9.42,1507780,1433947733.00,3.64,-4.15,-0.41,1.90
1542,2026-05-21,300015,9.41,9.36,9.63,9.24,1245961,1175989215.00,4.12,-1.16,-0.11,1.57
1543,2026-05-22,300015,9.33,9.17,9.40,9.17,770771,711638472.87,2.46,-2.03,-0.19,0.97



当前股票信息 开始拉取...
  ✅ 成功
  耗时: 3.78s
  行数: 9
  列数: 2

当前股票信息：


...


【经营现金流量净额】


,选项,指标,20260331,20251231,20250930,20250630,20250331,20241231,20240930,20240630,...,20100630,20100331,20091231,20090930,20090630,20090331,20081231,20080930,20071231,20061231
7,常用指标,经营现金流量净额,1708202556.08,5972727509.04,5078358713.04,3402066561.83,1820785553.45,4881686201.15,4298762594.82,2843399691.04,...,65236864.08,56567239.66,169137874.31,138838661.78,71018213.19,34742383.96,121174149.93,83903707.67,92187299.68,36904489.32



【毛利率】


,选项,指标,20260331,20251231,20250930,20250630,20250331,20241231,20240930,20240630,...,20100630,20100331,20091231,20090930,20090630,20090331,20081231,20080930,20071231,20061231
13,常用指标,毛利率,47.73,47.11,49.27,48.56,48.02,48.12,51.02,49.44,...,56.80,56.50,57.09,57.64,57.85,55.73,55.43,56.16,56.24,51.64



数据拉取完成
✅ 财务摘要
✅ 历史行情
✅ 当前股票信息
✅ 利润表
✅ 资产负债表
✅ 现金流量表

成功率: 6/6

DCF数据完整性
✅ 营收
✅ 净利润
✅ ROE
✅ 毛利率
✅ 历史股价
✅ 现金流
✅ 资产负债

脚本执行结束


In [ ]:
# ============================================================
# Cell 5：自动填充财务参数 + FCF 计算（多源降级，逐年 capex）
# ============================================================

import pandas as pd
import numpy as np

# ── 辅助：从 financial_abstract 宽表按指标名取最近 N 个年报值 ──
def extract_annual(df, metric, n=6):
    """返回 {year(int): value(float)} 升序字典，年报列名以 1231 结尾"""
    annual_cols = sorted(
        [c for c in df.columns if str(c).endswith('1231')],
        reverse=True,
    )
    row = df[df['指标'].astype(str).str.strip() == metric]
    if row.empty:
        return {}
    result = {}
    for col in annual_cols[:n]:
        try:
            val = float(row[col].values[0])
            if not np.isnan(val):
                result[int(str(col)[:4])] = val
        except Exception:
            pass
    return dict(sorted(result.items()))

# ── 辅助：从宽表（行=科目，列=期间）中按关键词匹配行，返回逐年字典 ──
def extract_annual_from_wide(df, keyword, item_col=None, n=6):
    """
    适配 stock_cash_flow_sheet_by_yearly_em / stock_financial_report_sina 等宽表：
    - item_col: 科目列名；None 时自动用第一列
    - 年份列自动识别（包含 4 位年份数字）
    返回 {year: value(元)} 字典
    """
    if df is None or df.empty:
        return {}
    if item_col is None:
        item_col = df.columns[0]
    year_cols = [c for c in df.columns if str(c).strip()[:4].isdigit()]
    year_cols = sorted(year_cols, reverse=True)[:n]
    row = df[df[item_col].astype(str).str.contains(keyword, na=False)]
    if row.empty:
        return {}
    result = {}
    for col in year_cols:
        try:
            val = float(str(row[col].values[0]).replace(',', ''))
            if not np.isnan(val):
                result[int(str(col).strip()[:4])] = val
        except Exception:
            pass
    return dict(sorted(result.items()))

# ════════════════════════════════════════════════════════════
# 1. 当前价格 & 股本
# ════════════════════════════════════════════════════════════
if not MANUAL_OVERRIDE and stock_info is not None:
    try:
        CURRENT_PRICE = float(stock_info[stock_info['item'] == '最新']['value'].values[0])
        TOTAL_SHARES  = float(stock_info[stock_info['item'] == '总股本']['value'].values[0]) / 1e8
        CURRENT_MV    = CURRENT_PRICE * TOTAL_SHARES
        print(f'✅ 实时价格：{CURRENT_PRICE:.2f}元  总股本：{TOTAL_SHARES:.4f}亿股  市值：{CURRENT_MV:.1f}亿元')
    except Exception as e:
        print(f'⚠️  价格/股本解析失败（{e}），请设置 MANUAL_OVERRIDE=True')
        CURRENT_MV = CURRENT_PRICE * TOTAL_SHARES
else:
    CURRENT_MV = CURRENT_PRICE * TOTAL_SHARES
    print(f'📌 手动参数：{CURRENT_PRICE:.2f}元  {TOTAL_SHARES:.4f}亿股  {CURRENT_MV:.1f}亿元')

# ════════════════════════════════════════════════════════════
# 2. 历史归母净利润（from financial_abstract）
# ════════════════════════════════════════════════════════════
if not MANUAL_OVERRIDE and financial_abstract is not None:
    _raw = extract_annual(financial_abstract, '归母净利润', n=6)
    if _raw:
        HIST_PROFIT = {y: v / 1e8 for y, v in _raw.items()}
        print(f'✅ 历史净利润（亿元）：{ {y: round(v,2) for y,v in HIST_PROFIT.items()} }')
    else:
        print('⚠️  未能解析归母净利润，请设置 MANUAL_OVERRIDE=True')
        HIST_PROFIT = {}
else:
    print(f'📌 手动净利润：{HIST_PROFIT}')

# ════════════════════════════════════════════════════════════
# 3. 历史经营现金流 OCF（from financial_abstract，逐年）
# ════════════════════════════════════════════════════════════
HIST_OCF = {}
if financial_abstract is not None:
    _ocf_raw = extract_annual(financial_abstract, '经营现金流量净额', n=6)
    if _ocf_raw:
        HIST_OCF = {y: v / 1e8 for y, v in _ocf_raw.items()}
        print(f'✅ 历史OCF（亿元）：   { {y: round(v,2) for y,v in HIST_OCF.items()} }')

# ════════════════════════════════════════════════════════════
# 4. 历史 Capex（逐年，三级降级）
#    Level-1：stock_cash_flow_sheet_by_yearly_em → 购建固定资产（精确）
#    Level-2：stock_financial_report_sina         → 购建固定资产（精确）
#    Level-3：financial_abstract 资本支出指标     → 若有
#    Level-4：单期 stock_cashflow 投资净现金流    → 仅用对应年份，其余年不扣
# ════════════════════════════════════════════════════════════
HIST_CAPEX = {}   # {year: 亿元}，正数
_capex_source = '无'

# Level-1
try:
    _cf_y = ak.stock_cash_flow_sheet_by_yearly_em(symbol=SYMBOL)
    # 列名示例：ITEM / 2024 / 2023 / 2022...  行含"购建固定资产"
    _item_col = _cf_y.columns[0]
    _cx = extract_annual_from_wide(_cf_y, '购建固定资产', item_col=_item_col, n=6)
    if _cx:
        HIST_CAPEX = {y: abs(v) / 1e8 for y, v in _cx.items()}
        _capex_source = 'stock_cash_flow_sheet_by_yearly_em（购建固定资产）'
except Exception as _e1:
    pass

# Level-2
if not HIST_CAPEX:
    try:
        _sym_sina = ('sh' if SYMBOL.startswith(('6','9')) else 'sz') + SYMBOL
        _cf_s = ak.stock_financial_report_sina(stock=_sym_sina, symbol='现金流量表')
        _item_col_s = _cf_s.columns[0]
        _cx2 = extract_annual_from_wide(_cf_s, '购建固定资产', item_col=_item_col_s, n=6)
        if _cx2:
            HIST_CAPEX = {y: abs(v) / 1e8 for y, v in _cx2.items()}
            _capex_source = 'stock_financial_report_sina（购建固定资产）'
    except Exception as _e2:
        pass

# Level-3：financial_abstract 里若有资本支出指标
if not HIST_CAPEX and financial_abstract is not None:
    for _kw in ['资本支出', '购建固定']:
        _cx3 = extract_annual(financial_abstract, _kw, n=6)
        if _cx3:
            HIST_CAPEX = {y: abs(v) / 1e8 for y, v in _cx3.items()}
            _capex_source = f'financial_abstract（{_kw}）'
            break

# Level-4：仅单期快照，只用最新年份
if not HIST_CAPEX and stock_cashflow is not None and not stock_cashflow.empty:
    try:
        _icf_col = [c for c in stock_cashflow.columns if '投资性现金流' in c and '净额' in c]
        if _icf_col:
            _icf_val = float(stock_cashflow[_icf_col[0]].values[0])
            # 只记最新一年（2024），其他年份不用错误扣除
            _latest_year = 2024
            HIST_CAPEX = {_latest_year: abs(_icf_val) / 1e8}
            _capex_source = f'stock_cashflow 投资净流出（仅{_latest_year}年，单期快照）'
    except Exception:
        pass

if HIST_CAPEX:
    print(f'✅ 历史Capex（亿元）：  { {y: round(v,2) for y,v in HIST_CAPEX.items()} }')
    print(f'   数据来源：{_capex_source}')
else:
    print('⚠️  无 Capex 数据，FCF 将等于 OCF（保守上界）')

# ════════════════════════════════════════════════════════════
# 5. FCF = OCF - Capex（逐年对齐）
# ════════════════════════════════════════════════════════════
HIST_FCF = {}
if HIST_OCF:
    for y, ocf in HIST_OCF.items():
        capex = HIST_CAPEX.get(y, 0)   # 没有对应年份 capex 则不扣（保守：FCF=OCF）
        HIST_FCF[y] = max(ocf - capex, 0)
    print(f'✅ 历史FCF（亿元）：   { {y: round(v,2) for y,v in HIST_FCF.items()} }')

# ════════════════════════════════════════════════════════════
# 6. FCF/净利润转换率（利润质量指标）
# ════════════════════════════════════════════════════════════
FCF_MARGIN = {}
avg_fcf_margin = 1.0
if HIST_PROFIT and HIST_FCF:
    common = sorted(set(HIST_PROFIT) & set(HIST_FCF))
    FCF_MARGIN = {y: HIST_FCF[y] / HIST_PROFIT[y] for y in common if HIST_PROFIT[y] > 0}
    avg_fcf_margin = float(np.mean(list(FCF_MARGIN.values()))) if FCF_MARGIN else 1.0
    print(f'\nFCF/净利润：{", ".join(f"{y}:{v:.2f}" for y,v in FCF_MARGIN.items())}')
    print(f'均值：{avg_fcf_margin:.2f}  （>1 现金充裕；<1 扩张期 capex 重或利润含水分）')

# ════════════════════════════════════════════════════════════
# 7. 派生估值变量
# ════════════════════════════════════════════════════════════
hist_years   = list(HIST_PROFIT.keys())
hist_profits = list(HIST_PROFIT.values())

hist_growth   = [(hist_profits[i]/hist_profits[i-1]-1)*100 for i in range(1, len(hist_profits))]
avg_hist_cagr = (hist_profits[-1]/hist_profits[0])**(1/(len(hist_profits)-1))-1 if len(hist_profits)>1 else 0

current_pe_ttm   = CURRENT_MV / hist_profits[-1] if hist_profits else 0
forecast_pe_2026 = CURRENT_MV / FORECAST_PROFIT.get(2026, hist_profits[-1]*1.2) if hist_profits else 0
forecast_pe_2027 = CURRENT_MV / FORECAST_PROFIT.get(2027, hist_profits[-1]*1.4) if hist_profits else 0
peg        = forecast_pe_2026 / (BASE_CAGR * 100) if BASE_CAGR else 0
peg_status = '低估' if peg < 1 else ('合理' if peg < 1.5 else ('偏贵' if peg < 2 else '高估'))

# ════════════════════════════════════════════════════════════
# 8. 概览输出
# ════════════════════════════════════════════════════════════
print()
print('=' * 54)
print(f'【{STOCK_NAME} 基本估值概览】')
print('=' * 54)
print(f'当前股价：          {CURRENT_PRICE:.2f} 元')
print(f'当前总市值：        {CURRENT_MV:.1f} 亿元')
if hist_profits:
    print(f'{hist_years[-1]}年净利润：       {hist_profits[-1]:.2f} 亿元')
if HIST_OCF:
    _oy = max(HIST_OCF)
    print(f'{_oy}年经营现金流：   {HIST_OCF[_oy]:.2f} 亿元')
if HIST_CAPEX:
    _cy = max(HIST_CAPEX)
    print(f'{_cy}年资本开支：     {HIST_CAPEX[_cy]:.2f} 亿元  [{_capex_source.split("（")[0]}]')
if HIST_FCF:
    _fy = max(HIST_FCF)
    print(f'{_fy}年FCF：          {HIST_FCF[_fy]:.2f} 亿元')
print(f'当前PE(TTM)：       {current_pe_ttm:.1f} 倍')
print(f'2026E PE：          {forecast_pe_2026:.1f} 倍')
print(f'2027E PE：          {forecast_pe_2027:.1f} 倍')
print(f'历史净利润CAGR：    {avg_hist_cagr*100:.1f}%')
print()
print('历史利润增速：')
for i, y in enumerate(hist_years[1:]):
    print(f'  {y}年: {hist_growth[i]:+.1f}%')
print(f'\nPEG（基准增速）：   {peg:.2f} → {peg_status}')


In [ ]:
# ============================================================
# Cell 5：自动填充财务参数 + FCF 计算
# ============================================================

import pandas as pd
import numpy as np

# ── 辅助：从 financial_abstract 宽表中按指标名取最近 N 个年报值 ──
def extract_annual(df, metric, n=5):
    """返回 {year(int): value(float)} 字典，取最近 n 个年报（列名以 1231 结尾）"""
    annual_cols = sorted(
        [c for c in df.columns if str(c).endswith('1231')],
        reverse=True
    )
    row = df[df['指标'].astype(str).str.strip() == metric]
    if row.empty:
        return {}
    result = {}
    for col in annual_cols[:n]:
        try:
            val = float(row[col].values[0])
            if not np.isnan(val):
                year = int(str(col)[:4])
                result[year] = val
        except Exception:
            pass
    return dict(sorted(result.items()))   # 升序

# ── 1. 当前价格 & 股本（来自 stock_info）──
if not MANUAL_OVERRIDE and stock_info is not None:
    try:
        _p = stock_info[stock_info['item'] == '最新']['value'].values[0]
        _s = stock_info[stock_info['item'] == '总股本']['value'].values[0]
        CURRENT_PRICE = float(_p)
        TOTAL_SHARES  = float(_s) / 1e8          # 股 → 亿股
        CURRENT_MV    = CURRENT_PRICE * TOTAL_SHARES
        print(f'✅ 实时价格：{CURRENT_PRICE:.2f}元  总股本：{TOTAL_SHARES:.4f}亿股  市值：{CURRENT_MV:.1f}亿元')
    except Exception as e:
        print(f'⚠️  价格/股本解析失败（{e}），请手动设置 MANUAL_OVERRIDE=True')
        CURRENT_PRICE = CURRENT_PRICE if 'CURRENT_PRICE' in dir() else 0
        TOTAL_SHARES  = TOTAL_SHARES  if 'TOTAL_SHARES'  in dir() else 1
        CURRENT_MV    = CURRENT_PRICE * TOTAL_SHARES
else:
    CURRENT_MV = CURRENT_PRICE * TOTAL_SHARES
    print(f'📌 手动参数：{CURRENT_PRICE:.2f}元  {TOTAL_SHARES:.4f}亿股  {CURRENT_MV:.1f}亿元')

# ── 2. 历史净利润（归母）——自动从 financial_abstract 解析 ──
if not MANUAL_OVERRIDE and financial_abstract is not None:
    _raw = extract_annual(financial_abstract, '归母净利润', n=6)
    if _raw:
        # 单位：元 → 亿元
        HIST_PROFIT = {y: v / 1e8 for y, v in _raw.items()}
        print(f'✅ 自动解析历史净利润（亿元）：{HIST_PROFIT}')
    else:
        print('⚠️  未能自动解析归母净利润，请手动设置 MANUAL_OVERRIDE=True')
        HIST_PROFIT = {}
else:
    print(f'📌 手动历史净利润：{HIST_PROFIT}')

# ── 3. 历史经营现金流（FCF 分子）──
HIST_OCF = {}   # 经营现金流（亿元）
HIST_FCF = {}   # 自由现金流 = OCF - Capex（亿元）

if financial_abstract is not None:
    _ocf_raw = extract_annual(financial_abstract, '经营现金流量净额', n=6)
    if _ocf_raw:
        HIST_OCF = {y: v / 1e8 for y, v in _ocf_raw.items()}
        print(f'✅ 经营现金流（亿元）：{HIST_OCF}')

# Capex 近似：从 stock_cashflow 的"投资性现金流-现金流量净额"取绝对值
# （实际 capex 只占投资现金流一部分，这里保守地用全部投资净流出作上限）
if stock_cashflow is not None and not stock_cashflow.empty:
    try:
        _icf_col = [c for c in stock_cashflow.columns if '投资性现金流' in c and '净额' in c]
        if _icf_col:
            _icf_val = float(stock_cashflow[_icf_col[0]].values[0])   # 元，通常为负
            _capex_billion = abs(_icf_val) / 1e8
            # 仅用最新一年 capex 估算历年 FCF（capex 数据只有单期快照）
            for y, ocf in HIST_OCF.items():
                HIST_FCF[y] = max(ocf - _capex_billion, 0)
            print(f'✅ 最新投资净现金流（绝对值作 capex 上限）：{_capex_billion:.1f}亿元')
            print(f'   FCF（保守）：{HIST_FCF}')
    except Exception as e:
        print(f'⚠️  Capex 解析失败（{e}），FCF 将等于 OCF')
        HIST_FCF = dict(HIST_OCF)
else:
    HIST_FCF = dict(HIST_OCF)

if not HIST_FCF and HIST_OCF:
    HIST_FCF = dict(HIST_OCF)

# ── 4. FCF 转换率（FCF / 净利润）——衡量利润质量 ──
FCF_MARGIN = {}
if HIST_PROFIT and HIST_FCF:
    common_years = sorted(set(HIST_PROFIT) & set(HIST_FCF))
    FCF_MARGIN = {y: HIST_FCF[y] / HIST_PROFIT[y] for y in common_years if HIST_PROFIT[y] > 0}
    avg_fcf_margin = np.mean(list(FCF_MARGIN.values())) if FCF_MARGIN else 1.0
    print(f'\nFCF/净利润（利润质量）：{", ".join(f"{y}:{v:.2f}" for y,v in FCF_MARGIN.items())}')
    print(f'近年均值：{avg_fcf_margin:.2f}  （>1 说明现金质量好；<1 说明利润含水分）')
else:
    avg_fcf_margin = 1.0

# ── 5. 派生估值变量 ──
hist_years   = list(HIST_PROFIT.keys())
hist_profits = list(HIST_PROFIT.values())

hist_growth   = [(hist_profits[i] / hist_profits[i-1] - 1) * 100 for i in range(1, len(hist_profits))]
avg_hist_cagr = (hist_profits[-1] / hist_profits[0]) ** (1 / (len(hist_profits) - 1)) - 1 if len(hist_profits) > 1 else 0

current_pe_ttm   = CURRENT_MV / hist_profits[-1]        if hist_profits else 0
forecast_pe_2026 = CURRENT_MV / FORECAST_PROFIT.get(2026, hist_profits[-1] * 1.2) if hist_profits else 0
forecast_pe_2027 = CURRENT_MV / FORECAST_PROFIT.get(2027, hist_profits[-1] * 1.4) if hist_profits else 0

peg        = forecast_pe_2026 / (BASE_CAGR * 100) if BASE_CAGR else 0
peg_status = '低估' if peg < 1 else ('合理' if peg < 1.5 else ('偏贵' if peg < 2 else '高估'))

# ── 6. 打印概览 ──
print()
print('=' * 52)
print(f'【{STOCK_NAME} 基本估值概览】')
print('=' * 52)
print(f'当前股价：         {CURRENT_PRICE:.2f} 元')
print(f'当前总市值：       {CURRENT_MV:.1f} 亿元')
if hist_profits:
    print(f'{hist_years[-1]}年净利润：      {hist_profits[-1]:.2f} 亿元')
if HIST_OCF:
    latest_ocf_year = max(HIST_OCF)
    print(f'{latest_ocf_year}年经营现金流：  {HIST_OCF[latest_ocf_year]:.2f} 亿元')
if HIST_FCF:
    latest_fcf_year = max(HIST_FCF)
    print(f'{latest_fcf_year}年FCF（保守）：  {HIST_FCF[latest_fcf_year]:.2f} 亿元')
print(f'当前PE(TTM)：      {current_pe_ttm:.1f} 倍')
print(f'2026E PE：         {forecast_pe_2026:.1f} 倍')
print(f'2027E PE：         {forecast_pe_2027:.1f} 倍')
print(f'历史净利润CAGR：   {avg_hist_cagr * 100:.1f}%')
print()
print('历史利润增速：')
for i, y in enumerate(hist_years[1:]):
    print(f'  {y}年: {hist_growth[i]:+.1f}%')
print(f'\nPEG（基准增速）：  {peg:.2f} → {peg_status}')


In [ ]:
# ============================================================
# Cell 6：反向DCF —— 当前市值隐含了什么增速预期？
# （净利润路径 & FCF 路径双轨并列）
# ============================================================

def reverse_dcf(current_mv, base_value, terminal_multiple, discount_rate, years):
    """
    反推隐含增速：current_mv*(1+r)^n = base*(1+g)^n * multiple
    base_value 可以是净利润（配 PE）或 FCF（配 P/FCF）
    """
    future_needed = current_mv * (1 + discount_rate) ** years
    terminal_needed = future_needed / terminal_multiple
    return (terminal_needed / base_value) ** (1 / years) - 1

base_profit_latest = hist_profits[-1]
base_fcf_latest    = HIST_FCF.get(max(HIST_FCF), base_profit_latest * avg_fcf_margin) if HIST_FCF else base_profit_latest

# 净利润路径（PE 终值）
implied_growth_pe = reverse_dcf(
    CURRENT_MV, base_profit_latest, TERMINAL_PE, DISCOUNT_RATE, FORECAST_YEARS
)

# FCF 路径（P/FCF 终值；成熟期 P/FCF 通常与 PE 接近，这里复用 TERMINAL_PE）
TERMINAL_PFCF = TERMINAL_PE   # 可独立调整
implied_growth_fcf = reverse_dcf(
    CURRENT_MV, base_fcf_latest, TERMINAL_PFCF, DISCOUNT_RATE, FORECAST_YEARS
) if base_fcf_latest > 0 else float('nan')

def growth_label(g):
    if g < BEAR_CAGR:   return '🟢', '市场定价极度悲观，存在明显低估'
    if g < BASE_CAGR:   return '🟡', '市场定价偏保守，有一定安全边际'
    if g < BULL_CAGR:   return '🟠', '市场定价合理，需增速超预期才有超额收益'
    return '🔴', '市场定价乐观，已超过乐观情景，上行空间有限'

color_pe,  assessment_pe  = growth_label(implied_growth_pe)
color_fcf, assessment_fcf = growth_label(implied_growth_fcf) if not np.isnan(implied_growth_fcf) else ('⚪', 'FCF数据不足')

print('=' * 55)
print('【反向DCF：市场隐含增速分析】')
print('=' * 55)
print(f'当前市值：    {CURRENT_MV:.1f} 亿元')
print(f'折现率：      {DISCOUNT_RATE*100:.0f}%    预测年限：{FORECAST_YEARS}年    成熟期PE：{TERMINAL_PE}x')
print()
print('─── 净利润路径 ───────────────────────────────')
print(f'基准净利润：  {base_profit_latest:.2f} 亿元（{hist_years[-1]}年）')
print(f'隐含增速：    {implied_growth_pe*100:.1f}%/年')
print(f'{color_pe}  {assessment_pe}')
print()
print('─── FCF 路径（更贴近真实价值）────────────────')
print(f'基准FCF：     {base_fcf_latest:.2f} 亿元（保守估算）')
if not np.isnan(implied_growth_fcf):
    print(f'隐含增速：    {implied_growth_fcf*100:.1f}%/年')
    print(f'{color_fcf}  {assessment_fcf}')
    if implied_growth_fcf > implied_growth_pe:
        print(f'  ↳ FCF隐含增速 > 净利润隐含增速：市场对现金流要求更高，利润质量受关注')
    else:
        print(f'  ↳ FCF隐含增速 < 净利润隐含增速：FCF优于净利润，现金质量好，PE低估了价值')
else:
    print('  FCF 数据不足，跳过')
print()
print('参考增速区间：')
print(f'  悲观 {BEAR_CAGR*100:.0f}%  |  基准 {BASE_CAGR*100:.0f}%  |  乐观 {BULL_CAGR*100:.0f}%  |  历史CAGR {avg_hist_cagr*100:.1f}%')

# 供后续 cell 引用
assessment = assessment_pe   # 净利润路径判断（Cell 11 摘要用）
implied_growth = implied_growth_pe


In [ ]:
# ============================================================
# Cell 7：三情景正向估值 —— 净利润路径 & FCF 路径双轨
# ============================================================

def scenario_valuation(base_value, cagr, terminal_multiple, discount_rate, years, current_mv):
    """
    正向 DCF：给定基础值、增速、终值倍数，计算折现后合理市值和年化回报
    base_value 可以是净利润或 FCF
    """
    future_val    = base_value * (1 + cagr) ** years
    terminal_mv   = future_val * terminal_multiple
    fair_mv_today = terminal_mv / (1 + discount_rate) ** years
    annual_return = (terminal_mv / current_mv) ** (1 / years) - 1
    return {
        'future_val':    future_val,
        'terminal_mv':   terminal_mv,
        'fair_mv_today': fair_mv_today,
        'annual_return': annual_return,
    }

cagr_map  = {'悲观': BEAR_CAGR, '基准': BASE_CAGR, '乐观': BULL_CAGR}
weights   = {'悲观': PROB_BEAR,  '基准': PROB_BASE,  '乐观': PROB_BULL}

# ── 净利润路径 ──
scenarios_pe = {
    name: scenario_valuation(base_profit_latest, g, TERMINAL_PE, DISCOUNT_RATE, FORECAST_YEARS, CURRENT_MV)
    for name, g in cagr_map.items()
}
weighted_return_pe  = sum(scenarios_pe[s]['annual_return']  * weights[s] for s in scenarios_pe)
weighted_fair_mv_pe = sum(scenarios_pe[s]['fair_mv_today'] * weights[s] for s in scenarios_pe)

# ── FCF 路径 ──
has_fcf = base_fcf_latest > 0
if has_fcf:
    scenarios_fcf = {
        name: scenario_valuation(base_fcf_latest, g, TERMINAL_PFCF, DISCOUNT_RATE, FORECAST_YEARS, CURRENT_MV)
        for name, g in cagr_map.items()
    }
    weighted_return_fcf  = sum(scenarios_fcf[s]['annual_return']  * weights[s] for s in scenarios_fcf)
    weighted_fair_mv_fcf = sum(scenarios_fcf[s]['fair_mv_today'] * weights[s] for s in scenarios_fcf)

# ── 打印净利润路径 ──
W = 62
def print_scenario_table(label, scenarios, weighted_fair_mv, weighted_return):
    print('=' * W)
    print(f'【三情景估值 — {label}】（{FORECAST_YEARS}年，折现率{DISCOUNT_RATE*100:.0f}%）')
    print('=' * W)
    print(f'{"情景":<4} {"增速":>5} {"概率":>5}  '
          f'{f"{FORECAST_YEARS}年后值":>8}  {"终值市值":>8}  {"折现合理市值":>10}  {"年化回报":>7}')
    print('-' * W)
    for name, s in scenarios.items():
        ret_str = f"{s['annual_return']*100:+.1f}%"
        print(f'{name:<4} {cagr_map[name]*100:>4.0f}%  {weights[name]*100:>4.0f}%  '
              f'{s["future_val"]:>7.1f}亿  {s["terminal_mv"]:>7.1f}亿  '
              f'{s["fair_mv_today"]:>9.1f}亿  {ret_str:>7}')
    print('-' * W)
    print(f'{"加权期望":<4} {"":>5} {"":>5}  {"":>8}  {"":>8}  '
          f'{weighted_fair_mv:>9.1f}亿  {weighted_return*100:>+6.1f}%')
    updown = (weighted_fair_mv / CURRENT_MV - 1) * 100
    print(f'\n当前市值：{CURRENT_MV:.1f}亿   加权合理市值：{weighted_fair_mv:.1f}亿   空间：{updown:+.1f}%')

print_scenario_table('净利润路径（PE终值）', scenarios_pe, weighted_fair_mv_pe, weighted_return_pe)

if has_fcf:
    print()
    print_scenario_table(f'FCF路径（P/FCF={TERMINAL_PFCF}x）', scenarios_fcf, weighted_fair_mv_fcf, weighted_return_fcf)
    print()
    print('─── 双路径对比 ──────────────────────────────────')
    print(f'{"":10} {"净利润路径":>10} {"FCF路径":>10}  说明')
    print(f'{"加权年化回报":10} {weighted_return_pe*100:>9.1f}% {weighted_return_fcf*100:>9.1f}%')
    fv_pe  = weighted_fair_mv_pe  / CURRENT_MV - 1
    fv_fcf = weighted_fair_mv_fcf / CURRENT_MV - 1
    print(f'{"合理市值空间":10} {fv_pe*100:>+9.1f}% {fv_fcf*100:>+9.1f}%')
    if weighted_fair_mv_fcf > weighted_fair_mv_pe:
        print('  ↳ FCF估值 > 净利润估值：现金质量好，净利润路径偏保守')
    else:
        print('  ↳ FCF估值 < 净利润估值：扩张期 capex 大，FCF路径更保守，建议参考 FCF 定价')

# 供后续 cell 引用（以净利润路径为主）
scenarios        = scenarios_pe
weighted_return  = weighted_return_pe
weighted_fair_mv = weighted_fair_mv_pe
updown           = (weighted_fair_mv / CURRENT_MV - 1) * 100

# 决策
if weighted_return >= 0.18:
    decision, dsymbol = '强烈推荐建仓，赔率极佳',        '🟢🟢'
elif weighted_return >= 0.12:
    decision, dsymbol = '可以建仓，赔率合理',             '🟢'
elif weighted_return >= 0.08:
    decision, dsymbol = '观望为主，等待更好买入点',       '🟡'
else:
    decision, dsymbol = '不建议买入，赔率不足',           '🔴'

print(f'\n{dsymbol} 综合决策（净利润路径）：{decision}')
print(f'   加权年化预期回报：{weighted_return*100:.1f}%')


In [ ]:
# ============================================================
# Cell 8：反推合理买入价 —— 要达到目标年化回报需要什么价格
# ============================================================

target_returns = [0.10, 0.15, 0.20, 0.25]
terminal_mv_base = scenarios['基准']['terminal_mv']

print('=' * 55)
print('【合理买入价区间（基准情景）】')
print('=' * 55)
print(f'基于基准增速 {BASE_CAGR*100:.0f}%，{FORECAST_YEARS}年后市值 = '
      f'{terminal_mv_base:.1f} 亿元')
print()
print(f'{"目标年化回报":<10} {"对应买入市值":>12} {"对应买入价":>10} {"较当前折价":>8}')
print('-' * 55)
for r in target_returns:
    buy_mv    = terminal_mv_base / (1 + r) ** FORECAST_YEARS
    buy_price = buy_mv / TOTAL_SHARES
    discount  = (buy_price / CURRENT_PRICE - 1) * 100
    marker    = ' ← 当前在此区间' if abs(discount) < 5 else ''
    print(f'{r*100:.0f}%{"":>8} {buy_mv:>10.1f}亿   {buy_price:>7.2f}元  {discount:>+6.1f}%{marker}')

print()
print(f'当前价格：{CURRENT_PRICE:.2f}元 / 市值：{CURRENT_MV:.1f}亿元')
print()
print('【各情景下获得15%年化回报的买入价】')
print('-' * 40)
for name, s in scenarios.items():
    buy_mv_15    = s['terminal_mv'] / (1.15) ** FORECAST_YEARS
    buy_price_15 = buy_mv_15 / TOTAL_SHARES
    disc_15      = (buy_price_15 / CURRENT_PRICE - 1) * 100
    print(f'{name}情景：{buy_price_15:.2f}元（市值{buy_mv_15:.1f}亿，较当前{disc_15:+.1f}%）')


In [ ]:
# ============================================================
# Cell 9：敏感性分析 —— 增速和PE假设对估值的影响
# ============================================================

import plotly.graph_objects as go
import numpy as np

cagr_range = np.arange(0.05, 0.45, 0.05)
pe_range   = [12, 15, 18, 20, 25, 30]

z_return = []
for pe in pe_range:
    row = []
    for g in cagr_range:
        future_profit = base_profit_latest * (1 + g) ** FORECAST_YEARS
        term_mv  = future_profit * pe
        ann_ret  = (term_mv / CURRENT_MV) ** (1 / FORECAST_YEARS) - 1
        row.append(round(ann_ret * 100, 1))
    z_return.append(row)

fig = go.Figure(data=go.Heatmap(
    z=z_return,
    x=[f'{g*100:.0f}%' for g in cagr_range],
    y=[f'{pe}x' for pe in pe_range],
    colorscale=[
        [0.0, '#d73027'], [0.3, '#f46d43'],
        [0.5, '#fee08b'], [0.65, '#a6d96a'],
        [0.8, '#1a9850'], [1.0, '#006837'],
    ],
    zmid=12,
    text=[[f'{v}%' for v in row] for row in z_return],
    texttemplate='%{text}',
    textfont={'size': 11},
    colorbar=dict(title='年化回报%'),
))

# 标记基准假设点
base_x_idx = int(min(range(len(cagr_range)), key=lambda i: abs(cagr_range[i] - BASE_CAGR)))
base_y_idx = pe_range.index(TERMINAL_PE) if TERMINAL_PE in pe_range else None
if base_y_idx is not None:
    fig.add_shape(
        type='rect',
        x0=base_x_idx - 0.5, x1=base_x_idx + 0.5,
        y0=base_y_idx - 0.5, y1=base_y_idx + 0.5,
        line=dict(color='blue', width=3),
    )

fig.update_layout(
    title=(f'{STOCK_NAME}（{SYMBOL}）敏感性分析 — 年化回报率热力图<br>'
           f'（当前市值{CURRENT_MV:.0f}亿，{FORECAST_YEARS}年维度，蓝框=基准假设）'),
    xaxis_title='利润年化增速（CAGR）',
    yaxis_title='成熟期终值PE',
    height=400,
)
fig.show()
print('绿色区域（>15%）= 值得买入；红色区域（<8%）= 不划算')


In [ ]:
# ============================================================
# Cell 10：利润增长轨迹可视化
# ============================================================

import plotly.graph_objects as go

last_hist_year     = max(HIST_PROFIT.keys())
last_forecast_year = max(FORECAST_PROFIT.keys())
pivot_profit       = FORECAST_PROFIT[last_forecast_year]

# 延伸年份（从机构预测末年往后补齐至 FORECAST_YEARS 总长度）
forecast_count = len(FORECAST_PROFIT)
ext_count      = FORECAST_YEARS - forecast_count
ext_years      = list(range(last_forecast_year + 1, last_forecast_year + ext_count + 1))

fig2 = go.Figure()

# 历史实际
fig2.add_trace(go.Scatter(
    x=list(HIST_PROFIT.keys()),
    y=list(HIST_PROFIT.values()),
    mode='lines+markers+text',
    name='历史实际',
    text=[f'{v:.1f}亿' for v in HIST_PROFIT.values()],
    textposition='top center',
    line=dict(color='#2196F3', width=2.5),
    marker=dict(size=8),
))

# 机构预测（从历史末年连线）
fig2.add_trace(go.Scatter(
    x=[last_hist_year] + list(FORECAST_PROFIT.keys()),
    y=[HIST_PROFIT[last_hist_year]] + list(FORECAST_PROFIT.values()),
    mode='lines+markers+text',
    name='机构预测',
    text=[None] + [f'{v:.1f}亿' for v in FORECAST_PROFIT.values()],
    textposition='top center',
    line=dict(color='#2196F3', width=2, dash='dash'),
    marker=dict(size=7, symbol='diamond'),
))

# 三情景延伸（从机构预测末年开始）
scenario_cfg = [
    ('悲观', BEAR_CAGR, '#FF5252'),
    ('基准', BASE_CAGR, '#FF9800'),
    ('乐观', BULL_CAGR, '#4CAF50'),
]
for label, cagr_val, color in scenario_cfg:
    ext_profits = [pivot_profit * (1 + cagr_val) ** i for i in range(1, ext_count + 1)]
    fig2.add_trace(go.Scatter(
        x=[last_forecast_year] + ext_years,
        y=[pivot_profit] + ext_profits,
        mode='lines',
        name=f'{label}（{cagr_val*100:.0f}%）',
        line=dict(color=color, width=1.5, dash='dot'),
    ))

fig2.update_layout(
    title=f'{STOCK_NAME}（{SYMBOL}）净利润增长轨迹与三情景展望',
    xaxis_title='年份',
    yaxis_title='归母净利润（亿元）',
    height=420,
    legend=dict(x=0.02, y=0.98),
)
fig2.show()


In [ ]:
# ============================================================
# Cell 11：一页式投资摘要
# ============================================================

W = 60
print('=' * W)
print(f'  {STOCK_NAME}（{SYMBOL}）投资摘要')
print('=' * W)
print(f'  当前价格：{CURRENT_PRICE:.2f}元  |  市值：{CURRENT_MV:.1f}亿元')
print(f'  PE(TTM)：{current_pe_ttm:.1f}x  |  2026E PE：{forecast_pe_2026:.1f}x  |  2027E PE：{forecast_pe_2027:.1f}x')
print(f'  PEG（基准增速）：{peg:.2f}  →  {peg_status}')
print('-' * W)

# 利润质量
if FCF_MARGIN:
    latest_fcf_y = max(FCF_MARGIN)
    print(f'  FCF/净利润（{latest_fcf_y}）：{FCF_MARGIN[latest_fcf_y]:.2f}  近年均值：{avg_fcf_margin:.2f}')
print(f'  市场隐含增速（净利润路径）：{implied_growth_pe*100:.1f}%  →  {assessment_pe}')
if not np.isnan(implied_growth_fcf):
    print(f'  市场隐含增速（FCF路径）：    {implied_growth_fcf*100:.1f}%  →  {assessment_fcf}')
print('-' * W)

# 三情景加权
print(f'  加权年化回报（净利润路径）：{weighted_return_pe*100:.1f}%')
if has_fcf:
    print(f'  加权年化回报（FCF路径）：    {weighted_return_fcf*100:.1f}%')
fv_pe = (weighted_fair_mv_pe / CURRENT_MV - 1) * 100
print(f'  加权合理市值：{weighted_fair_mv_pe:.0f}亿元  （较当前{fv_pe:+.1f}%）')
print('-' * W)

# 买入价
print('  目标年化 → 对应买入价（基准情景，净利润路径）：')
for r in [0.10, 0.15, 0.20]:
    buy_mv    = scenarios['基准']['terminal_mv'] / (1 + r) ** FORECAST_YEARS
    buy_price = buy_mv / TOTAL_SHARES
    disc      = (buy_price / CURRENT_PRICE - 1) * 100
    arrow     = ' ← 当前在此区间' if -5 < disc < 5 else ''
    print(f'  {r*100:.0f}%年化 → {buy_price:.1f}元（{disc:+.1f}%）{arrow}')
print('-' * W)
print(f'  {dsymbol} 综合决策：{decision}')
print('=' * W)
print('  ⚠️  本工具仅做辅助分析，不构成投资建议')


---
## 使用说明

**分析新股票只需修改 Cell 3 的少量参数：**

| 参数 | 说明 | 是否需要手动填 |
|------|------|----------------|
| `SYMBOL` | 股票代码 | ✅ 必填 |
| `STOCK_NAME` | 股票名称 | ✅ 必填 |
| `FORECAST_PROFIT` | 机构预测利润（亿元） | ✅ 必填（同花顺F10→盈利预测） |
| `BEAR/BASE/BULL_CAGR` | 三情景增速假设 | ✅ 必填（你的判断） |
| `TERMINAL_PE` | 成熟期PE | ✅ 建议调整（制造15-20，消费20-25，科技25-30） |
| `DISCOUNT_RATE` | 你的回报率要求 | 可选，默认12% |
| `CURRENT_PRICE` / `TOTAL_SHARES` / `HIST_PROFIT` | 历史财务数据 | **自动填充**，无需手动输入 |

> 若自动填充失败，在 Cell 3 设置 `MANUAL_OVERRIDE = True` 并手动填写三行覆盖值。

---

**核心逻辑：**

- **反向DCF（双路径）**：净利润路径 + FCF路径，分别反推市场隐含增速，两者对比揭示利润质量
- **三情景正向估值**：概率加权计算「期望年化回报」，双路径并列
- **FCF/净利润比**：> 1 说明利润质量高；< 1 说明扩张期 capex 重，PE 可能高估价值
- **热力图**：增速 × 终值PE 的每种组合对应的回报率，直观呈现假设敏感性
- **合理买入价**：要达到目标回报，需要在什么价位买入

**为什么用 FCF 而不只用净利润？**

爱尔眼科等重资产扩张型公司，净利润包含大量折旧前利润，实际可分配现金（FCF）更低。
FCF = 经营现金流 − 资本开支（本工具用投资净现金流绝对值作保守上限）。
FCF路径下的估值更保守，但更接近真实股东价值。
